# 03 ML Low Rating Prediction

Notebook ini membuat baseline Machine Learning untuk memprediksi risiko order mendapat review rendah (`review_score <= 2`). Analisis bersifat prediktif/asosiatif dan digunakan sebagai pendukung Customer Experience monitoring.

## 1. Setup

Notebook memakai fungsi dari `scripts/train_low_rating_model.py` agar logika training konsisten dengan script produksi lokal.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(repo_root))

from scripts.train_low_rating_model import (
    load_dataset,
    print_dataset_overview,
    train_models,
    save_feature_importance_chart,
    MODEL_PATH,
    METRICS_PATH,
)


## 2. Load Data

Data diambil dari ClickHouse mart. Query hanya membaca data dan memfilter order yang memiliki review valid.

In [ ]:
df = load_dataset()
df.head()

## 3. Dataset Overview

Cek shape, distribusi target, missing value, dan fitur yang tersedia.

In [ ]:
print_dataset_overview(df)

## 4. Train Baseline Models

Model yang dibandingkan: Logistic Regression dan Random Forest. Karena low rating adalah kelas minoritas, metrik recall, precision, F1-score, dan ROC-AUC perlu dibaca bersama.

In [ ]:
result = train_models(df)
result["best_model_name"]

## 5. Metrics Summary

In [ ]:
import pandas as pd

metrics_table = pd.DataFrame(
    {
        model_name: {
            key: value
            for key, value in model_metrics.items()
            if key in ["accuracy", "precision", "recall", "f1_score", "roc_auc"]
        }
        for model_name, model_metrics in result["metrics"].items()
    }
).T
metrics_table

## 6. Feature Importance

Feature importance digunakan sebagai sinyal prediktif, bukan bukti kausal.

In [ ]:
feature_importance = result["feature_importance"]
feature_importance.head(20)

In [ ]:
save_feature_importance_chart(feature_importance)
print("Feature importance chart saved if feature importance is available.")

## 7. Save Model

Model artifact disimpan lokal dan tidak perlu ikut commit jika berbentuk binary `.pkl`.

In [ ]:
import json
import joblib

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(result["best_model"], MODEL_PATH)

metrics_payload = {
    "best_model": result["best_model_name"],
    "target_distribution": result["target_distribution"],
    "numerical_features": result["numerical_features"],
    "categorical_features": result["categorical_features"],
    "metrics": result["metrics"],
    "top_feature_importance": result["feature_importance"].head(20).to_dict(orient="records"),
}

with open(METRICS_PATH, "w", encoding="utf-8") as file:
    json.dump(metrics_payload, file, indent=2)

print(f"Saved model to {MODEL_PATH}")
print(f"Saved metrics to {METRICS_PATH}")

## 8. Interpretation Notes

- Model ini cocok untuk post-delivery risk monitoring atau pre-review intervention.
- Fitur delivery seperti `delay_days`, `delivery_status`, dan `delay_bucket` tersedia setelah proses pengiriman, sehingga belum cocok untuk prediksi saat order baru dibuat.
- Hasil model perlu dibaca bersama dashboard root cause dan bukan sebagai klaim kausal absolut.